# 01 — 2D Scan

Define a phase transition model, run the BubbleMaster 2D scan over γ and time,
and cache results for bootstrapping in notebook 02.

Set `USE_GPU = True` to generate SLURM scripts for a multi-node GPU job
instead of running locally on CPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ptbuilder as pt
from ptbuilder.ic import _two_bubble_ic, _cutoff_times

## 1. Define the model

In [ ]:
# Choose potential and create PhysicsModel
lambda_bar = 0.069

# Set to True to generate GPU SLURM scripts instead of running locally
USE_GPU = True

potential = pt.Phi4Potential(lambda_bar)
config    = pt.Config()          # uses default data/ directory and bin/ binaries
model     = pt.PhysicsModel(potential, config)

REPO_ROOT = Path(config.bubblemaster_bin).parent.parent

print(f"Model name:   {model.name}")
print(f"Results dir:  {model.results_dir}")
print(f"phi_false:    {potential.phi_false}")
print(f"phi_true:     {potential.phi_true}")
print(f"Delta V:      {potential.delta_V:.6f}")
print(f"USE_GPU:      {USE_GPU}")

## 2. Compute the instanton profile (disk-cached)

In [ ]:
profile = model.instanton

print(f"rin_0:   {profile.rin_0:.4f}")
print(f"rmid_0:  {profile.rmid_0:.4f}")
print(f"rout_0:  {profile.rout_0:.4f}")
print(f"wall width: {profile.wall_width:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(profile.R, profile.Phi)
plt.axvline(profile.rin_0,  ls='--', color='C1', label='r_in')
plt.axvline(profile.rmid_0, ls='--', color='C2', label='r_mid')
plt.axvline(profile.rout_0, ls='--', color='C3', label='r_out')
plt.xlabel('r')
plt.ylabel(r'$\phi$')
plt.title('Instanton profile')
plt.legend()
plt.tight_layout()

## 3. Define the scan grid

In [ ]:
n_gamma  = 32
n_t      = 32

gammas   = np.linspace(1., 8., n_gamma)
times    = np.linspace(1., 70., n_t)

bm_params = pt.BubbleMasterParams(
    dz           = 0.005,
    n_w          = 128,
    n_k          = 51,
    how_often_ds = 5,
    baby_steps   = 100,
    cutoff_type  = 0,
)

scan_cfg = pt.ScanConfig(
    gammas    = gammas,
    times     = times,
    bm_params = bm_params,
)

print(f"Gammas: {gammas[[0,-1]]}  ({n_gamma} points)")
print(f"Times:  {times[[0,-1]]}  ({n_t} points)")

## 4. Run the scan

**CPU** (`USE_GPU=False`): runs locally; each (γ, model) pair is cached in
`data/<model>/scan_cache.h5` — re-running only computes missing entries.

**GPU** (`USE_GPU=True`): prints a timing preview, writes setup files, and
generates a SLURM batch script that runs all γ values simultaneously
(one GPU per γ, 4 per node). Submit the script, then run *§ 4b* to load results.

In [ ]:
if not USE_GPU:
    # ------------------------------------------------------------------ CPU
    config.check_binaries()
    scan_results = pt.run_scan(model, scan_cfg, config)
    print(f"\nScan complete: {len(scan_results)} gamma values")

else:
    # ------------------------------------------------------------------ GPU
    # Timing preview: d and effective t_max per gamma.
    # Reference fit from lb=0.069 validation; adjust _g_ref/_t_ref for other models.
    _g_ref = np.array([3.873, 5.9])
    _t_ref = np.array([135.,  219.])
    _exp   = np.log(_t_ref[1]/_t_ref[0]) / np.log(_g_ref[1]/_g_ref[0])
    _A     = _t_ref[0] / _g_ref[0]**_exp

    t_max_global = float(times[-1])
    print(f"Power-law exponent: {_exp:.3f}")
    print(f"\n{'\u03b3':>8}  {'d':>8}  {'t_nat':>8}  {'t_eff':>8}  {'est (s)':>9}  note")
    print('-' * 60)
    est_wall = []
    for gamma in gammas:
        _, _, d, _ = _two_bubble_ic(profile, gamma, bm_params.dz)
        _, _, _, t_nat = _cutoff_times(d, bm_params.cutoff_type, bm_params.t_0_scal,
                                       t_min_global=0.)
        t_eff  = max(t_nat, t_max_global)
        t_est  = _A * gamma**_exp * (t_eff / t_nat)
        note   = 'forced' if t_eff > t_nat + 0.01 else 'natural'
        est_wall.append(t_est)
        print(f"{gamma:8.4f}  {d:8.2f}  {t_nat:8.2f}  {t_eff:8.2f}  {t_est:9.0f}  {note}")
    print(f"\nExpected wall time (slowest): {max(est_wall)/60:.1f} min")

In [ ]:
if USE_GPU:
    # Write one setup file per gamma
    scan_dir = model.results_dir / 'scan'
    scan_dir.mkdir(parents=True, exist_ok=True)

    setup_paths, out_dirs = [], []
    for gamma in gammas:
        setup_path = scan_dir / f'setup_gamma{gamma:.6g}.h5'
        out_dir    = scan_dir / f'out_gamma{gamma:.6g}'
        out_dir.mkdir(parents=True, exist_ok=True)
        pt.write_2d_setup(model, gamma=gamma, times=times,
                          path=setup_path, params=bm_params)
        setup_paths.append(str(setup_path.resolve()))
        out_dirs.append(str(out_dir.resolve()))

    print(f"Wrote {len(setup_paths)} setup files to {scan_dir}")

In [ ]:
if USE_GPU:
    # SLURM settings — set walltime based on timing preview above
    WALLTIME = '01:00:00'
    ACCOUNT  = 'm3166_g'
    QOS      = 'regular'

    n_tasks = n_gamma
    n_nodes = n_tasks // 4
    assert n_tasks % 4 == 0, f"n_gamma ({n_tasks}) must be divisible by 4 (GPUs per node)"

    scripts_dir  = REPO_ROOT / 'scripts'
    logs_dir     = REPO_ROOT / 'logs'
    scripts_dir.mkdir(exist_ok=True)
    logs_dir.mkdir(exist_ok=True)

    model_tag    = model.name.replace(' ', '_')
    wrapper_path = scripts_dir / f'scan_{model_tag}_wrapper.sh'
    slurm_path   = scripts_dir / f'scan_{model_tag}.sh'
    gpu_bin      = str(config.bubblemaster_gpu_bin.resolve())

    setup_arr = ' '.join(f'"{p}"' for p in setup_paths)
    out_arr   = ' '.join(f'"{o}"' for o in out_dirs)

    wrapper_path.write_text(f"""#!/bin/bash
SETUPS=({setup_arr})
OUTDIRS=({out_arr})

IDX=$SLURM_PROCID
exec {gpu_bin} "${{SETUPS[$IDX]}}" "${{OUTDIRS[$IDX]}}/"
""")
    wrapper_path.chmod(0o755)

    slurm_path.write_text(f"""#!/bin/bash
#SBATCH --job-name=scan_{model_tag}
#SBATCH --output={logs_dir}/scan_{model_tag}_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos={QOS}
#SBATCH -A {ACCOUNT}
#SBATCH --nodes={n_nodes}
#SBATCH --ntasks={n_tasks}
#SBATCH --ntasks-per-node=4
#SBATCH --gpus-per-task=1
#SBATCH --time={WALLTIME}

srun --ntasks={n_tasks} {wrapper_path}
""")

    print(f'SLURM script: {slurm_path}')
    print(f'Wrapper:      {wrapper_path}')
    print(f'Submit with:  sbatch {slurm_path}')

## 4b. Load GPU results  *(run after SLURM job completes)*

Reads each γ output directory and merges into `scan_cache.h5`.
Skip when `USE_GPU = False`.

In [ ]:
if USE_GPU:
    from ptbuilder.scan import _save_scan_cache

    scan_results = {}
    missing      = []

    for gamma, out_dir in zip(gammas, out_dirs):
        out_path = Path(out_dir)
        if not list(out_path.glob('result_*.h5')):
            missing.append(gamma)
            continue
        scan_results[float(gamma)] = pt.load_bm_result(out_path, gamma, times)

    print(f"Loaded: {len(scan_results)}/{n_gamma} gamma points")
    if missing:
        print(f"Missing output for \u03b3 = {[f'{g:.4g}' for g in missing]}")

    cache_path = model.results_dir / 'scan_cache.h5'
    _save_scan_cache(scan_results, cache_path)
    print(f"Cached \u2192 {cache_path}")

## 5. Inspect results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

i_gamma_show = len(gammas) // 2
gamma_show   = gammas[i_gamma_show]
res          = scan_results[float(gamma_show)]

# Cumulative spectrum at each time
ax = axes[0]
for i_t in range(n_t):
    ax.plot(res.w, res.spectrum[i_t], alpha=0.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('w')
ax.set_ylabel('spectrum')
ax.set_title(f'Cumulative spectrum  (\u03b3={gamma_show:.1f})')

# dE/dt heatmap
ax = axes[1]
dEdt_show = np.array([scan_results[float(g)].dEdt[i_gamma_show]
                       for g in gammas])
ax.imshow(dEdt_show, aspect='auto', origin='lower', cmap='afmhot')
ax.set_xlabel('k index')
ax.set_ylabel('gamma index')
ax.set_title('dE/dt(\u03b3, k)')

plt.tight_layout()